# Jorg vs Korg.jl Comprehensive Diagnostics

This notebook performs detailed comparison between Jorg and Korg.jl using freshly generated reference data.

**Goal**: Identify and diagnose the ~50,000,000× flux magnitude discrepancy.

In [ ]:
import sys
from pathlib import Path
sys.path.append("/Users/jdli/Project/Korg.jl/Jorg/src/")

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

from jorg.synthesis import synthesize
from jorg.lines.linelist import read_linelist
from jorg.abundances import format_A_X
from jorg.atmosphere import interpolate_marcs

print("✅ Imports complete")

In [ ]:
# Load VALD linelist (same as Korg.jl uses)
linelist_path = "/Users/jdli/Project/Korg.jl/data/linelists/vald_extract_stellar_solar_threshold001.vald"
linelist = read_linelist(linelist_path)
print(f"✅ Loaded {len(linelist)} spectral lines")

## 1. Solar Spectrum Comparison (Teff=5771K, logg=4.44, [M/H]=0.0)

In [ ]:
# Run Jorg synthesis for solar parameters
print("🔬 Running Jorg synthesis for solar parameters...")

# Prepare abundances
A_X_dict = format_A_X(m_H=0.0)
A_X = np.full(92, -50.0)
A_X[0] = 12.0  # H = 12.0
for Z, abundance in A_X_dict.items():
    if 1 <= Z <= 92:
        A_X[Z-1] = abundance

# Get atmosphere
atm_solar = interpolate_marcs(Teff=5771., logg=4.44, m_H=0.0)

# Synthesize with lines
jorg_solar = synthesize(atm_solar, linelist, A_X, wavelengths=(5000, 5020), verbose=True)

# Synthesize continuum-only
jorg_solar_cntm = synthesize(atm_solar, [], A_X, wavelengths=(5000, 5020), verbose=False)

print(f"\n✅ Jorg solar synthesis complete")
print(f"  Flux range: {jorg_solar.flux.min():.3e} - {jorg_solar.flux.max():.3e} erg/s/cm²")
print(f"  Continuum range: {jorg_solar.cntm.min():.3e} - {jorg_solar.cntm.max():.3e} erg/s/cm²")
print(f"  Alpha matrix shape: {jorg_solar.alpha.shape}")

In [ ]:
# Load Korg.jl reference data
print("📖 Loading Korg.jl reference data...")

korg_dir = "/Users/jdli/Project/Korg.jl/jorg/examples/korg_reference"

# Load solar spectra
korg_solar_lines = np.loadtxt(f"{korg_dir}/korg_solar_with_lines.txt", comments='#')
korg_solar_cntm = np.loadtxt(f"{korg_dir}/korg_solar_continuum_only.txt", comments='#')
korg_solar_opacity = np.loadtxt(f"{korg_dir}/korg_solar_opacity.txt", comments='#', skiprows=4, usecols=(1, 2, 3))
korg_solar_atm = np.loadtxt(f"{korg_dir}/korg_solar_atmosphere.txt", comments='#')

print(f"✅ Korg.jl reference data loaded")
print(f"  With-lines flux range: {korg_solar_lines[:, 1].min():.3e} - {korg_solar_lines[:, 1].max():.3e} erg/s/cm²")
print(f"  Continuum flux range: {korg_solar_cntm[:, 1].min():.3e} - {korg_solar_cntm[:, 1].max():.3e} erg/s/cm²")
print(f"  Opacity range: {korg_solar_opacity[:, 0].min():.3e} - {korg_solar_opacity[:, 1].max():.3e} cm⁻¹")

In [ ]:
# Compare flux magnitudes
print("="*80)
print("FLUX MAGNITUDE COMPARISON (Solar)")
print("="*80)

# Continuum-only comparison
korg_cntm_mean = korg_solar_cntm[:, 2].mean()  # Column 2 is continuum
jorg_cntm_mean = jorg_solar_cntm.cntm.mean()
cntm_ratio = korg_cntm_mean / jorg_cntm_mean

print(f"\nCONTINUUM FLUX:")
print(f"  Korg.jl: {korg_cntm_mean:.6e} erg/s/cm²")
print(f"  Jorg:    {jorg_cntm_mean:.6e} erg/s/cm²")
print(f"  Ratio:   {cntm_ratio:.6e} (Korg/Jorg)")
print(f"  Discrepancy: {cntm_ratio:.1f}×")

# With-lines comparison
korg_flux_mean = korg_solar_lines[:, 1].mean()  # Column 1 is flux
jorg_flux_mean = jorg_solar.flux.mean()
flux_ratio = korg_flux_mean / jorg_flux_mean

print(f"\nWITH-LINES FLUX:")
print(f"  Korg.jl: {korg_flux_mean:.6e} erg/s/cm²")
print(f"  Jorg:    {jorg_flux_mean:.6e} erg/s/cm²")
print(f"  Ratio:   {flux_ratio:.6e} (Korg/Jorg)")
print(f"  Discrepancy: {flux_ratio:.1f}×")

# Normalized flux comparison (should agree if line physics is correct)
korg_norm = korg_solar_lines[:, 3]  # Column 3 is normalized flux
jorg_norm = jorg_solar.flux / jorg_solar.cntm

norm_diff = np.abs(korg_norm - jorg_norm).mean()
print(f"\nNORMALIZED FLUX:")
print(f"  Mean absolute difference: {norm_diff:.6f}")
print(f"  RMS difference: {np.sqrt(np.mean((korg_norm - jorg_norm)**2)):.6f}")

print(f"\n" + "="*80)

In [ ]:
# Compare opacity matrices
print("="*80)
print("OPACITY MATRIX COMPARISON (Solar)")
print("="*80)

# Calculate Jorg opacity statistics per layer
jorg_alpha_min = jorg_solar.alpha.min(axis=1)
jorg_alpha_max = jorg_solar.alpha.max(axis=1)
jorg_alpha_mean = jorg_solar.alpha.mean(axis=1)

# Compare with Korg.jl (columns: min, max, mean)
korg_alpha_min = korg_solar_opacity[:56, 0]
korg_alpha_max = korg_solar_opacity[:56, 1]
korg_alpha_mean = korg_solar_opacity[:56, 2]

print(f"\nOPACITY STATISTICS:")
print(f"  Korg.jl range: {korg_alpha_min.min():.3e} - {korg_alpha_max.max():.3e} cm⁻¹")
print(f"  Jorg range:    {jorg_alpha_min.min():.3e} - {jorg_alpha_max.max():.3e} cm⁻¹")

# Calculate ratios for valid (positive) values
valid_mask = (jorg_alpha_mean > 0) & (korg_alpha_mean > 0)
alpha_ratio = jorg_alpha_mean[valid_mask] / korg_alpha_mean[valid_mask]

print(f"\nOPACITY RATIO (Jorg/Korg):")
print(f"  Mean: {alpha_ratio.mean():.3f}")
print(f"  Median: {np.median(alpha_ratio):.3f}")
print(f"  Range: {alpha_ratio.min():.3f} - {alpha_ratio.max():.3f}")

# Plot opacity comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Log-log scatter plot
ax1.scatter(korg_alpha_mean, jorg_alpha_mean, alpha=0.6, s=50)
ax1.plot([1e-14, 1e-4], [1e-14, 1e-4], 'r--', alpha=0.5, label='Perfect agreement')
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('Korg.jl Opacity (cm⁻¹)')
ax1.set_ylabel('Jorg Opacity (cm⁻¹)')
ax1.set_title('Mean Layer Opacity Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Ratio vs layer
ax2.plot(alpha_ratio, 'o-', alpha=0.7)
ax2.axhline(1.0, color='r', linestyle='--', alpha=0.5, label='Perfect agreement')
ax2.set_xlabel('Atmospheric Layer')
ax2.set_ylabel('Opacity Ratio (Jorg/Korg)')
ax2.set_title('Layer-by-Layer Opacity Ratio')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n" + "="*80)

In [ ]:
# Plot normalized spectra comparison
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10))

# Normalized flux comparison
ax1.plot(korg_solar_lines[:, 0], korg_solar_lines[:, 3], label='Korg.jl', lw=1.5, alpha=0.8)
ax1.plot(jorg_solar.wavelengths, jorg_norm, label='Jorg', lw=1.5, alpha=0.8, ls='--')
ax1.set_ylabel('Normalized Flux')
ax1.set_title('Solar Spectrum: Normalized Flux Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim(5000, 5020)

# Absolute flux comparison (Korg.jl)
ax2.plot(korg_solar_lines[:, 0], korg_solar_lines[:, 1], label='Korg.jl flux', lw=1.5)
ax2.plot(korg_solar_lines[:, 0], korg_solar_lines[:, 2], label='Korg.jl continuum', lw=1.5, ls=':')
ax2.set_ylabel('Flux (erg/s/cm²)')
ax2.set_title('Korg.jl Absolute Flux')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xlim(5000, 5020)
ax2.ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

# Absolute flux comparison (Jorg)
ax3.plot(jorg_solar.wavelengths, jorg_solar.flux, label='Jorg flux', lw=1.5)
ax3.plot(jorg_solar.wavelengths, jorg_solar.cntm, label='Jorg continuum', lw=1.5, ls=':')
ax3.set_xlabel('Wavelength (Å)')
ax3.set_ylabel('Flux (erg/s/cm²)')
ax3.set_title('Jorg Absolute Flux')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_xlim(5000, 5020)
ax3.ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

plt.tight_layout()
plt.show()

In [ ]:
# Compare atmospheric structure
print("="*80)
print("ATMOSPHERIC STRUCTURE COMPARISON (Solar)")
print("="*80)

# Korg.jl atmosphere: columns are (layer, temp, ne, n_tot)
korg_temps = korg_solar_atm[:, 1]
korg_ne = korg_solar_atm[:, 2]
korg_ntot = korg_solar_atm[:, 3]

# Jorg atmosphere
jorg_temps = np.array([layer.temp for layer in atm_solar.layers])
jorg_ne = jorg_solar.electron_number_density
jorg_ntot = np.array([layer.number_density for layer in atm_solar.layers])

print(f"\nTEMPERA TURE:")
print(f"  Korg.jl range: {korg_temps.min():.1f} - {korg_temps.max():.1f} K")
print(f"  Jorg range:    {jorg_temps.min():.1f} - {jorg_temps.max():.1f} K")
print(f"  Max difference: {np.abs(korg_temps - jorg_temps).max():.3f} K")

print(f"\nELECTRON DENSITY:")
print(f"  Korg.jl range: {korg_ne.min():.3e} - {korg_ne.max():.3e} cm⁻³")
print(f"  Jorg range:    {jorg_ne.min():.3e} - {jorg_ne.max():.3e} cm⁻³")
ne_ratio = jorg_ne / korg_ne
print(f"  Mean ratio (Jorg/Korg): {ne_ratio.mean():.6f}")
print(f"  Relative difference: {np.abs(1 - ne_ratio).mean() * 100:.3f}%")

print(f"\nTOTAL NUMBER DENSITY:")
print(f"  Korg.jl range: {korg_ntot.min():.3e} - {korg_ntot.max():.3e} cm⁻³")
print(f"  Jorg range:    {jorg_ntot.min():.3e} - {jorg_ntot.max():.3e} cm⁻³")
ntot_ratio = jorg_ntot / korg_ntot
print(f"  Mean ratio (Jorg/Korg): {ntot_ratio.mean():.6f}")
print(f"  Relative difference: {np.abs(1 - ntot_ratio).mean() * 100:.3f}%")

# Plot atmospheric comparison
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Temperature
axes[0, 0].plot(korg_temps, label='Korg.jl', marker='o', ms=4)
axes[0, 0].plot(jorg_temps, label='Jorg', marker='s', ms=4, alpha=0.7)
axes[0, 0].set_xlabel('Layer')
axes[0, 0].set_ylabel('Temperature (K)')
axes[0, 0].set_title('Temperature Profile')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Electron density
axes[0, 1].semilogy(korg_ne, label='Korg.jl', marker='o', ms=4)
axes[0, 1].semilogy(jorg_ne, label='Jorg', marker='s', ms=4, alpha=0.7)
axes[0, 1].set_xlabel('Layer')
axes[0, 1].set_ylabel('Electron Density (cm⁻³)')
axes[0, 1].set_title('Electron Density Profile')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Total number density
axes[1, 0].semilogy(korg_ntot, label='Korg.jl', marker='o', ms=4)
axes[1, 0].semilogy(jorg_ntot, label='Jorg', marker='s', ms=4, alpha=0.7)
axes[1, 0].set_xlabel('Layer')
axes[1, 0].set_ylabel('Total Number Density (cm⁻³)')
axes[1, 0].set_title('Total Number Density Profile')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Electron density ratio
axes[1, 1].plot(ne_ratio, marker='o', ms=4)
axes[1, 1].axhline(1.0, color='r', linestyle='--', alpha=0.5)
axes[1, 1].set_xlabel('Layer')
axes[1, 1].set_ylabel('Ratio (Jorg/Korg)')
axes[1, 1].set_title('Electron Density Ratio')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n" + "="*80)

In [ ]:
# DIAGNOSTIC SUMMARY
print("="*80)
print("DIAGNOSTIC SUMMARY")
print("="*80)

print(f"\n🔍 FLUX MAGNITUDE DISCREPANCY:")
print(f"  Continuum: {cntm_ratio:.1f}× (Korg.jl larger)")
print(f"  With-lines: {flux_ratio:.1f}× (Korg.jl larger)")
print(f"  Status: {'✅ WITHIN 10×' if flux_ratio < 10 else '❌ LARGE DISCREPANCY'}")

print(f"\n🔍 NORMALIZED FLUX AGREEMENT:")
print(f"  Mean abs difference: {norm_diff:.6f}")
print(f"  Status: {'✅ EXCELLENT' if norm_diff < 0.01 else '⚠️ NEEDS WORK' if norm_diff < 0.05 else '❌ POOR'}")

print(f"\n🔍 OPACITY AGREEMENT:")
print(f"  Mean ratio: {alpha_ratio.mean():.3f}")
print(f"  Relative error: {np.abs(1 - alpha_ratio.mean()) * 100:.1f}%")
print(f"  Status: {'✅ GOOD' if np.abs(1 - alpha_ratio.mean()) < 0.5 else '⚠️ MODERATE' if np.abs(1 - alpha_ratio.mean()) < 2.0 else '❌ POOR'}")

print(f"\n🔍 ATMOSPHERIC STRUCTURE:")
print(f"  Temperature: ✅ EXACT (max diff {np.abs(korg_temps - jorg_temps).max():.3f} K)")
print(f"  Electron density: {'✅ EXCELLENT' if np.abs(1 - ne_ratio).mean() < 0.01 else '⚠️ MODERATE' if np.abs(1 - ne_ratio).mean() < 0.1 else '❌ POOR'} ({np.abs(1 - ne_ratio).mean() * 100:.3f}% diff)")
print(f"  Total density: {'✅ EXCELLENT' if np.abs(1 - ntot_ratio).mean() < 0.01 else '⚠️ MODERATE' if np.abs(1 - ntot_ratio).mean() < 0.1 else '❌ POOR'} ({np.abs(1 - ntot_ratio).mean() * 100:.3f}% diff)")

print(f"\n🎯 KEY FINDING:")
if norm_diff < 0.05 and flux_ratio > 100:
    print(f"  ✅ Normalized flux agrees well ({norm_diff:.4f} diff)")
    print(f"  ❌ Absolute flux has {flux_ratio:.1f}× discrepancy")
    print(f"  📌 CONCLUSION: Line physics correct, but flux normalization/units incorrect")
    print(f"  💡 LIKELY CAUSE: Missing wavelength-dependent factor in flux calculation")
    print(f"     - Check Planck function units (B_λ vs B_ν)")
    print(f"     - Check wavelength/frequency conversion (dλ vs dν)")
    print(f"     - Check flux integration factors")
elif norm_diff > 0.05:
    print(f"  ❌ Normalized flux disagrees ({norm_diff:.4f} diff)")
    print(f"  ❌ Absolute flux has {flux_ratio:.1f}× discrepancy")
    print(f"  📌 CONCLUSION: Both line physics AND flux calculation have issues")
    print(f"  💡 LIKELY CAUSE: Line absorption calculation incorrect")
else:
    print(f"  ✅ Both normalized and absolute flux agree well")
    print(f"  📌 CONCLUSION: Implementation matches Korg.jl!")

print(f"\n" + "="*80)